In [4]:
import os
import pandas as pd

In [5]:
print(os.getcwd())

c:\Users\selin\Desktop\job-market-analysis\etl


Setup & Load

In [10]:
df = pd.read_csv("../data/raw/kumarijob_info.csv")

Profile raw data

In [11]:
df.head()

,title,company,location,job_url,scraped_date,category
0,Sales Executive,Classic Tech Pvt. Ltd.,"Chabahil, Suryabinayak, Golfutar,Kalanki",https://www.kumarijob.com/classictech-pvt-ltd/...,3/10/2026,it
1,Fiber Bike Rider,Classic Tech Pvt. Ltd.,"kalanki, thankot, balaju,lalitpur,thimi",https://www.kumarijob.com/classictech-pvt-ltd/...,3/10/2026,it
2,Senior Data Engineer,CloudFactory,"Kathmandu, Nepal",https://www.kumarijob.com/more-jobs/67578-seni...,3/10/2026,it
3,Mid-Level SEO Analyst,Varosa Technology,"Kathmandu, Nepal",https://www.kumarijob.com/more-jobs/67577-mid-...,3/10/2026,it
4,Business and Data Analyst,Abacus Insights,"Kathmandu, Nepal",https://www.kumarijob.com/more-jobs/67576-busi...,3/10/2026,it


In [12]:
df.dtypes

title           str
company         str
location        str
job_url         str
scraped_date    str
category        str
dtype: object

In [13]:
df.isnull().sum()

title            0
company          1
location        10
job_url          1
scraped_date     0
category         0
dtype: int64

Remove Duplicates


In [14]:
len(df)



188

In [15]:
df=df.drop_duplicates(subset=["title","company"])

In [16]:
len(df)

188

Remove junk rows 

In [22]:
JUNK_TITLES=[
    "similar openings",
    "simialar jobs",
    "you may also like",
    "featured jobs"

]
before=len(df)
df=df[~df["title"].str.lower().str.strip().isin(JUNK_TITLES)]
after=len(df)

The ~ operator means "NOT" in pandas — it inverts the filter. df[condition] keeps matching rows, df[~condition] keeps everything else.

In [23]:
print(f"Removed{before-after} junk rows")
print(f"Remaining:{after}rows")

Removed0 junk rows
Remaining:187rows


Remocce Duplicate


In [25]:
before=len(df)

df=df.drop_duplicates(subset=["title","company"])

after=len(df)
print(f"Remaining:{after} rows")

Remaining:187 rows


In [27]:
import re 

In [28]:
def clean_title(title):
    if pd.isna(title):
        return "Unknown"
    
    title = str(title).strip()
    
    # Remove extra info after dash
    # "Data Center Technician - NP - Lalitpur - On-site" → "Data Center Technician"
    title = re.sub(r"\s*-\s*(NP|Nepal|Remote|On-site|Onsite|Hybrid).*", 
                   "", title, flags=re.IGNORECASE)
    
    # Remove content inside brackets
    # "Credit Analyst (CA)" → "Credit Analyst"
    title = re.sub(r"\s*\(.*?\)", "", title)
    
    # Normalize seniority prefixes
    title = re.sub(r"\bSr\.?\b", "Senior", title, flags=re.IGNORECASE)
    title = re.sub(r"\bJr\.?\b", "Junior", title, flags=re.IGNORECASE)
    
    # Title case
    title = title.strip().title()
    
    return title

df["title_clean"] = df["title"].apply(clean_title)

# Show what changed
changed = df[df["title"] != df["title_clean"]][["title", "title_clean"]]
print(f"Titles cleaned: {len(changed)}")
display(changed.head(15))

Titles cleaned: 65


,title,title_clean
3,Mid-Level SEO Analyst,Mid-Level Seo Analyst
4,Business and Data Analyst,Business And Data Analyst
7,Data Center Technician - NP - Lalitpur - On-site,Data Center Technician
8,Data Center Technician - NP - Thapathali - On-...,Data Center Technician
9,Data Center Technician - NP - Kathmandu - On-site,Data Center Technician
11,SEO Expert,Seo Expert
13,Senior SEO Specialist / Executive,Senior Seo Specialist / Executive
14,Software QA Engineer 2,Software Qa Engineer 2
16,Data and Platform Manager,Data And Platform Manager
17,Finance and Operation Officer,Finance And Operation Officer
